# Follow-up experiments: rotation, PR sweep, N/D sweep

Runs the three experiments from `NOTES.md` §4g in the order `NOTES.md` §6 asks for.

**Before running anything, read `PREDICTIONS.md`.** It was written before these
runs and states what each config should show. If you look at results first, the
pre-registration is worthless.

Total cost is ~63x one `nd_D4_N31` run (the `--plan` cell below prints the
breakdown). Time the first config and multiply — that is the only estimate that
transfers across GPUs.

Set the accelerator to **GPU T4 x1** (or P100) in the notebook settings.

In [ ]:
import os, pathlib, subprocess, sys, torch

# Point this at the repo. On Kaggle, add it as a dataset or clone it here.
REPO = pathlib.Path('/kaggle/working/icl-unlearning')
if not REPO.exists():
    !git clone https://github.com/manyagupta13/icl-unlearning.git {REPO}
os.chdir(REPO)

print('cwd     ', os.getcwd())
print('torch   ', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print('gpu     ', p.name, f'{p.total_memory/1e9:.1f} GB')

## 1. Tests first

`NOTES.md` §5 flags that the test suite was written in an environment without
torch and never executed there. Same is true of the batch runner. Run both
before spending GPU hours.

In [ ]:
!python -m pytest tests/ -q -x --ignore=tests/verify_batch_preflight.py
!python tests/verify_batch_preflight.py

## 2. Generate configs and cost the batch

In [ ]:
!python scripts/make_configs.py --out configs/generated
!python scripts/prereg_table.py
!python scripts/run_all_experiments.py --plan

## 3. The gate: the null control

`rot_mid_identity` gives all three groups the same spectrum *and* the same
(identity) basis, so they are the same distribution — `full` and `oracle` train
on indistinguishable data. **The AUC must sit at chance.** If it does not, the
two hypotheses are coupled somewhere and nothing else in the batch is
interpretable (`PREDICTIONS.md` P8).

The runner enforces this and stops on failure. Run this cell on its own first.

In [ ]:
!python scripts/run_all_experiments.py --only rot_mid_identity

## 4. The batch

`--resume` skips any config whose artifacts already exist, so re-running this
cell after a dropped session picks up where it left off. Per-config logs go to
`artifacts/logs/`, machine-readable status to `artifacts/BATCH_STATUS.json`.

If the session limit is a problem, run one tier per session:
`--tier rotation`, then `--tier pr`, then `--tier nd`.

In [ ]:
!python scripts/run_all_experiments.py --resume

## 5. Health check

`check_ensemble_health` *prints* rather than raises on a Frobenius-norm outlier
(NOTES §4c), which is invisible in 16 configs of scrolling output. The runner
collects them; this re-prints them so they cannot be missed.

**A warning here means: do not interpret that config.**

In [ ]:
import json
st = json.load(open('artifacts/BATCH_STATUS.json'))
for k, v in st.items():
    if v.get('warnings'):
        print(k)
        for line in v['warnings']:
            print('   ', line)
    if 'null_check' in v:
        print(f"null check {k}: {'PASS' if v['null_check']['pass'] else 'FAIL'}"
              f"  {v['null_check']['detail']}")
print('\nconfigs finished:', sum('sweep' in v['stages'] for v in st.values()))

## 6. Package results for download

The `.pt` ensembles are large and reproducible from the configs; the CSVs and
figures are what you want off the machine.

In [ ]:
!mkdir -p /kaggle/working/out
!cp artifacts/results_*.csv artifacts/results_*_config.json /kaggle/working/out/ 2>/dev/null
!cp artifacts/BATCH_STATUS.json /kaggle/working/out/
!cp -r figures /kaggle/working/out/figures
!cd /kaggle/working && tar czf followups_results.tar.gz out && ls -lh followups_results.tar.gz

## 7. Score the pre-registration

Fill in the table in `PREDICTIONS.md` §8 from the CSVs. Predictions that failed
go in the write-up as failures — that is the entire point of having written
them down first.